In [5]:
import sys
import os
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV

## Training Baseline Models (Huiyu)

#### In this notebook, I train and compare seveal baseline models using the preprocessed dataset generated from `Preprocessing.py`.

In [6]:
project_root = os.path.abspath("..")
sys.path.append(project_root)
print(project_root)

# import the preprocessing function
from src.Preprocessing import get_preprocessed_data

# load preprocessed data, this function returns train/test split data
X_train, X_test, y_train, y_test = get_preprocessed_data(split=True)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

c:\Users\yhy_s\Downloads\california-housing


TypeError: get_preprocessed_data() got an unexpected keyword argument 'split'

In [ ]:
print("Feature columns:")
print(X_train.columns)

# sanity check: target should not be inside features
assert "ClosePrice" not in X_train.columns

print("\nFeature dtypes:")
print(X_train.dtypes)

# checking if there are any remaining missing values that need to be handled
na_counts = X_train.isna().sum()
na_counts = na_counts[na_counts > 0].sort_values(ascending=False)
print("\nMissing values per column:")
print(na_counts)

Feature columns:
Index(['Latitude', 'Longitude', 'PostalCode', 'StreetNumberNumeric',
       'AttachedGarageYN', 'BathroomsTotalInteger', 'BedroomsTotal',
       'FireplaceYN', 'GarageSpaces', 'LivingArea', 'MainLevelBedrooms',
       'NewConstructionYN', 'ParkingTotal', 'PoolPrivateYN', 'Stories',
       'ViewYN', 'YearBuilt', 'LotSizeAcres', 'LotSizeArea',
       'LotSizeSquareFeet', 'AssociationFee', 'DaysOnMarket',
       'sin_closed_date', 'Flooring_Stone', 'Flooring_Tile',
       'Flooring_SeeRemarks', 'Flooring_Bamboo', 'Flooring_Brick',
       'Flooring_Wood', 'Flooring_Laminate', 'Flooring_Vinyl',
       'Flooring_Concrete', 'Flooring_Carpet', 'Levels_One', 'Levels_Two',
       'Levels_MultiSplit', 'Levels_ThreeOrMore'],
      dtype='object')

Feature dtypes:
Latitude                 float64
Longitude                float64
PostalCode                 int64
StreetNumberNumeric      float64
AttachedGarageYN         boolean
BathroomsTotalInteger    float64
BedroomsTotal          

## Handle missing values

#### As we can see above, some features still comtian missing values after preprocessing.

#### I chose to use median imputation to fill missing values.

In [ ]:
# Used median imputation for missing values.
imputer = SimpleImputer(strategy="median")

X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Missing values after imputation (train):",
      X_train_imputed.isna().sum().sum())
print("Missing values after imputation (test):",
      X_test_imputed.isna().sum().sum())

Missing values after imputation (train): 0
Missing values after imputation (test): 0


#### Define evaluation metrics

#### I evaluate the baseline models using RMSE and R-square.

In [ ]:
def evaluate_model(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    print(f"{name}")
    print("RMSE:", rmse)
    print("R2:", r2)
    print("-"*30)

#### Train baseline models

In this step, I chose the following models as the baseline starting point:
- Linear Regression
- Ridge Regression
- Random Forest
- XGBoost

In [ ]:
# Linear Regression baseline

lin_model = LinearRegression()
lin_model.fit(X_train_imputed, y_train)
lin_pred = lin_model.predict(X_test_imputed)
evaluate_model("Linear Regression", y_test, lin_pred)


# Ridge Regression baseline

ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_imputed, y_train)
ridge_pred = ridge_model.predict(X_test_imputed)
evaluate_model("Ridge Regression", y_test, ridge_pred)


# Random Forest baseline

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_imputed, y_train)
rf_pred = rf_model.predict(X_test_imputed)
evaluate_model("Random Forest", y_test, rf_pred)


# XGBoost baseline

xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror"
)
xgb_model.fit(X_train_imputed, y_train)
xgb_pred = xgb_model.predict(X_test_imputed)
evaluate_model("XGBoost", y_test, xgb_pred)

Linear Regression
RMSE: 0.3315319080663181
R2: 0.5290250186979062
------------------------------
Ridge Regression
RMSE: 0.33152784050495865
R2: 0.5290365754008155
------------------------------


c:\Users\yhy_s\miniforge3\envs\dsc80\Lib\site-packages\sklearn\linear_model\_ridge.py:216: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.0638047519929094e-18.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


Random Forest
RMSE: 0.1744497070588015
R2: 0.8695969767072564
------------------------------
XGBoost
RMSE: 0.17583930268286635
R2: 0.8675112273389889
------------------------------


#### Baseline results summary

##### As we can see from the results, tree-based models perform much better than linear models.

##### Linear / Ridge models reach around 0.53 R-square. Random Forest and XGBoost reach around 0.87 R-square. This shows that the relationship between features and ClosePrice is highly non-linear.


#### Tuning Random Forest

In [ ]:
rf_candidates = [
    {
        "model_name": "RF_baseline_reference",
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
    },
    {
        "model_name": "RF_more_trees",
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
    },
    {
        "model_name": "RF_shallower_trees",
        "n_estimators": 200,
        "max_depth": 20,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
    },
    {
        "model_name": "RF_regularized_split",
        "n_estimators": 200,
        "max_depth": 20,
        "min_samples_split": 10,
        "min_samples_leaf": 4,
    },
    {
        "model_name": "RF_stronger_regularization",
        "n_estimators": 300,
        "max_depth": 15,
        "min_samples_split": 10,
        "min_samples_leaf": 4,
    },
]

rf_tuning_results = []
best_rf_model = None
best_rf_result = None
best_r2 = -np.inf

for config in rf_candidates:
    rf_tuned_model = RandomForestRegressor(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        min_samples_split=config["min_samples_split"],
        min_samples_leaf=config["min_samples_leaf"],
        random_state=42,
        n_jobs=-1
    )

    rf_tuned_model.fit(X_train_imputed, y_train)
    rf_tuned_pred = rf_tuned_model.predict(X_test_imputed)

    rmse = np.sqrt(mean_squared_error(y_test, rf_tuned_pred))
    r2 = r2_score(y_test, rf_tuned_pred)

    result_row = {
        "model_name": config["model_name"],
        "n_estimators": config["n_estimators"],
        "max_depth": config["max_depth"],
        "min_samples_split": config["min_samples_split"],
        "min_samples_leaf": config["min_samples_leaf"],
        "RMSE": rmse,
        "R2": r2
    }

    rf_tuning_results.append(result_row)

    print(f'Model: {config["model_name"]}')
    print(f'Parameters: {config}')
    print("RMSE:", rmse)
    print("R2:", r2)
    print("-" * 50)

    if r2 > best_r2:
        best_r2 = r2
        best_rf_model = rf_tuned_model
        best_rf_result = result_row


rf_results_df = pd.DataFrame(rf_tuning_results)
rf_results_df = rf_results_df.sort_values(by="R2", ascending=False).reset_index(drop=True)

print("Random Forest tuning summary:")
display(rf_results_df)


print("Best Random Forest configuration:")
print(best_rf_result)

best_rf_pred = best_rf_model.predict(X_test_imputed)

print("\nBest tuned Random Forest performance:")
evaluate_model("Best Tuned Random Forest", y_test, best_rf_pred)

Model: RF_baseline_reference
Parameters: {'model_name': 'RF_baseline_reference', 'n_estimators': 200, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
RMSE: 0.1744497070588015
R2: 0.8695969767072564
--------------------------------------------------
Model: RF_more_trees
Parameters: {'model_name': 'RF_more_trees', 'n_estimators': 300, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
RMSE: 0.17410392982311393
R2: 0.8701134087584207
--------------------------------------------------
Model: RF_shallower_trees
Parameters: {'model_name': 'RF_shallower_trees', 'n_estimators': 200, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 1}
RMSE: 0.17545310614161433
R2: 0.8680925594548425
--------------------------------------------------
Model: RF_regularized_split
Parameters: {'model_name': 'RF_regularized_split', 'n_estimators': 200, 'max_depth': 20, 'min_samples_split': 10, 'min_samples_leaf': 4}
RMSE: 0.17189695282730566
R2: 0.8733854757318325
-------

,model_name,n_estimators,max_depth,min_samples_split,min_samples_leaf,RMSE,R2
0,RF_regularized_split,200,20.0,10,4,0.171897,0.873385
1,RF_more_trees,300,NaN,2,1,0.174104,0.870113
2,RF_baseline_reference,200,NaN,2,1,0.174450,0.869597
3,RF_shallower_trees,200,20.0,2,1,0.175453,0.868093
4,RF_stronger_regularization,300,15.0,10,4,0.180254,0.860775


Best Random Forest configuration:
{'model_name': 'RF_regularized_split', 'n_estimators': 200, 'max_depth': 20, 'min_samples_split': 10, 'min_samples_leaf': 4, 'RMSE': 0.17189695282730566, 'R2': 0.8733854757318325}

Best tuned Random Forest performance:
Best Tuned Random Forest
RMSE: 0.17189695282730566
R2: 0.8733854757318325
------------------------------


#### Tuning XGBoost

In [ ]:
xgb_candidates = [
    {
        "model_name": "XGB_baseline_reference",
        "n_estimators": 200,
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "model_name": "XGB_more_trees",
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "model_name": "XGB_lower_learning_rate",
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "model_name": "XGB_shallower_trees",
        "n_estimators": 200,
        "max_depth": 4,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "model_name": "XGB_more_regularized_sampling",
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.7,
        "colsample_bytree": 0.7,
    },
]



xgb_tuning_results = []
best_xgb_model = None
best_xgb_result = None
best_xgb_r2 = -np.inf

for config in xgb_candidates:
    xgb_tuned_model = XGBRegressor(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        learning_rate=config["learning_rate"],
        subsample=config["subsample"],
        colsample_bytree=config["colsample_bytree"],
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror"
    )

    xgb_tuned_model.fit(X_train_imputed, y_train)
    xgb_tuned_pred = xgb_tuned_model.predict(X_test_imputed)

    rmse = np.sqrt(mean_squared_error(y_test, xgb_tuned_pred))
    r2 = r2_score(y_test, xgb_tuned_pred)

    result_row = {
        "model_name": config["model_name"],
        "n_estimators": config["n_estimators"],
        "max_depth": config["max_depth"],
        "learning_rate": config["learning_rate"],
        "subsample": config["subsample"],
        "colsample_bytree": config["colsample_bytree"],
        "RMSE": rmse,
        "R2": r2
    }

    xgb_tuning_results.append(result_row)

    print(f'Model: {config["model_name"]}')
    print(f'Parameters: {config}')
    print("RMSE:", rmse)
    print("R2:", r2)
    print("-" * 50)

    if r2 > best_xgb_r2:
        best_xgb_r2 = r2
        best_xgb_model = xgb_tuned_model
        best_xgb_result = result_row



xgb_results_df = pd.DataFrame(xgb_tuning_results)
xgb_results_df = xgb_results_df.sort_values(by="R2", ascending=False).reset_index(drop=True)

print("XGBoost tuning summary:")
display(xgb_results_df)



print("Best XGBoost configuration:")
print(best_xgb_result)

best_xgb_pred = best_xgb_model.predict(X_test_imputed)

print("\nBest tuned XGBoost performance:")
evaluate_model("Best Tuned XGBoost", y_test, best_xgb_pred)

Model: XGB_baseline_reference
Parameters: {'model_name': 'XGB_baseline_reference', 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8}
RMSE: 0.17583930268286635
R2: 0.8675112273389889
--------------------------------------------------
Model: XGB_more_trees
Parameters: {'model_name': 'XGB_more_trees', 'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8}
RMSE: 0.17096622726682215
R2: 0.8747528571106562
--------------------------------------------------
Model: XGB_lower_learning_rate
Parameters: {'model_name': 'XGB_lower_learning_rate', 'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8}
RMSE: 0.17921056579868885
R2: 0.8623822689404925
--------------------------------------------------
Model: XGB_shallower_trees
Parameters: {'model_name': 'XGB_shallower_trees', 'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.1, 'subsample': 0.8, 'col

,model_name,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,RMSE,R2
0,XGB_more_trees,300,6,0.10,0.8,0.8,0.170966,0.874753
1,XGB_baseline_reference,200,6,0.10,0.8,0.8,0.175839,0.867511
2,XGB_lower_learning_rate,300,6,0.05,0.8,0.8,0.179211,0.862382
3,XGB_shallower_trees,200,4,0.10,0.8,0.8,0.195874,0.835601
4,XGB_more_regularized_sampling,300,4,0.05,0.7,0.7,0.203389,0.822744


Best XGBoost configuration:
{'model_name': 'XGB_more_trees', 'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'RMSE': 0.17096622726682215, 'R2': 0.8747528571106562}

Best tuned XGBoost performance:
Best Tuned XGBoost
RMSE: 0.17096622726682215
R2: 0.8747528571106562
------------------------------


#### GridSearchCV for Random Forest

In [ ]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

rf_grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    verbose=2
)

rf_grid.fit(X_train_imputed, y_train)

print("Best parameters:", rf_grid.best_params_)
print("Best CV score:", rf_grid.best_score_)

Fitting 3 folds for each of 48 candidates, totalling 144 fits
Best parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best CV score: 0.8539935018002094


In [ ]:
best_rf = rf_grid.best_estimator_
rf_tuned_pred = best_rf.predict(X_test_imputed)

evaluate_model("Tuned Random Forest", y_test, rf_tuned_pred)

Tuned Random Forest
RMSE: 0.19226588461511715
R2: 0.8416012929643923
------------------------------


In [ ]:
import shap

X_shap = X_test_imputed.iloc[:300]

explainer = shap.TreeExplainer(best_rf)

shap_val = explainer.shap_values(X_shap)

shap.summary_plot(shap_val, X_shap)